In [1]:
import sqlite3
import pandas as pd
import os
from mappings import resolve_company_sec, fetch_sec_tickers, get_biotech_universe, clean_name
from scraper import fetch_trials
import requests

save_dir = 'C:\\Users\\chris\Documents\\personal_projects'
os.chdir(save_dir)

conn = sqlite3.connect("biotech.db")

df = pd.read_sql_query("""
    WITH latest_trial AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY nct_id ORDER BY snapshot_date DESC) AS rn
        FROM trials
    ),
    latest_pub AS (
        SELECT nct_id, title AS latest_publication, pub_date,
               ROW_NUMBER() OVER (PARTITION BY nct_id ORDER BY pub_date DESC) AS rn
        FROM publications
    )
    SELECT t.company, t.nct_id, t.phase, t.status, t.conditions,
           t.primary_completion_date, t.enrollment,
           p.latest_publication, p.pub_date
    FROM latest_trial t
    LEFT JOIN latest_pub p
        ON t.nct_id = p.nct_id AND p.rn = 1
    WHERE t.rn = 1
      AND t.phase IN ('PHASE3', 'PHASE4')
      AND t.status = 'ACTIVE_NOT_RECRUITING'
      AND t.primary_completion_date BETWEEN date('now') AND date('now', '+6 months')
    ORDER BY t.primary_completion_date
""", conn)

print(len(df))
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
df.sort_values('primary_completion_date')[['company', 'phase', 'primary_completion_date', 'conditions', 'enrollment', 'nct_id', 'latest_publication']].head(20)
#pd.reset_option('display.max_colwidth')
#pd.reset_option('display.max_rows')

135


,company,phase,primary_completion_date,conditions,enrollment,nct_id,latest_publication
0,amgen,PHASE4,2026-05-22,"STEMI - ST Elevation Myocardial Infarction,NSTEMI - Non-ST Segment Elevation MI",2166,NCT04951856,Optimal Medical Therapy Targeting Lipids and Inflammation for Secondary Prevention in Patients Undergoing Percutaneous Coronary Intervention.
1,novartis,PHASE3,2026-05-22,Chronic Inducible Urticaria,362,NCT05976243,Emerging IgE and non-IgE targeted therapies for chronic urticaria.
2,astrazeneca,PHASE3,2026-05-29,Hepatocellular Carcinoma,908,NCT03847428,None
3,astrazeneca,PHASE3,2026-05-29,Severe Eosinophilic Asthma,504,NCT06465485,None
4,abbvie,PHASE3,2026-05-31,Crohn's Disease,1336,NCT03105102,Comparative Efficacy and Safety of Advanced Therapies in Maintenance Treatment of Adult Patients with Moderate-to-Severe Crohn's Disease: A Systematic Literature Review and Network Meta-Analysis.
5,crispr therapeutics,PHASE3,2026-05-31,"Sickle Cell Disease,Hydroxyurea Failure,Hydroxyurea Intolerance,Hemoglobinopathies,Hematological Diseases",13,NCT05329649,Clinical Trial Landscape of Gene-Edited Autologous Hematopoietic Stem Cells for Hemoglobinopathies and Immunodeficiencies.
6,crispr therapeutics,PHASE3,2026-05-31,"Beta-Thalassemia,Thalassemia,Genetic Diseases, Inborn,Hematologic Diseases,Hemoglobinopathies",16,NCT05356195,Clinical Trial Landscape of Gene-Edited Autologous Hematopoietic Stem Cells for Hemoglobinopathies and Immunodeficiencies.
7,exelixis,PHASE3,2026-05-31,Colorectal Cancer,901,NCT05425940,"TYRO3, AXL, MERTK and Their Ligands in Brain Metastases From Colorectal Cancers."
8,eli lilly,PHASE3,2026-05-31,"Type 2 Diabetes,Obesity,Overweight,Obstructive Sleep Apnea",1000,NCT05929079,Beyond Glycemic Control: GLP-1RA-Based Therapies and Emerging Targets Beyond the Metabolic Axis.
9,biontech,PHASE3,2026-05-31,Metastatic Breast Cancer,541,NCT06018337,Antibodies to watch in 2026.
